# Práctica de laboratorio · Optimización de hiperparámetros en regresión logística

**FP13 · Inteligencia Artificial** —  Universidad Autónoma de Guadalajara  
**Unidad 5:** Clasificación y regresión con aprendizaje supervisado (5.5 Regresión logística · 5.7 Matrices de confusión) — con adelanto de **6.5 Búsqueda de rejilla**

---

### 🏥 Contexto clínico
El servicio de cardiología de un hospital quiere una herramienta de apoyo: a partir de 11 variables clínicas de rutina (edad, tipo de dolor torácico, colesterol, ECG en reposo, frecuencia cardiaca máxima, depresión del segmento ST, etc.) debe estimar si un paciente **tiene enfermedad cardiaca** y, por lo tanto, si conviene enviarlo a estudios especializados.

En clases anteriores ya entrenaste regresiones logísticas **con los hiperparámetros por defecto**. Hoy vas a responder tres preguntas:

1. ¿Qué tan bueno es realmente el modelo base, medido con métricas **clínicas** (sensibilidad, especificidad, F1…)?
2. ¿Mejora el modelo si **buscamos sistemáticamente** los mejores hiperparámetros con `GridSearchCV`?
3. ¿Qué pasa con los pacientes enfermos que no detectamos si **movemos el umbral de decisión**?

### 🎯 Objetivos de aprendizaje
Al terminar podrás:
- Calcular e interpretar **sensibilidad, especificidad, precisión, F1 y AUC** con `scikit-learn`.
- Diseñar desde cero un **diccionario de hiperparámetros** para `LogisticRegression` dentro de un `Pipeline`.
- Configurar `GridSearchCV` con validación cruzada estratificada y **elegir la métrica a optimizar** con criterio clínico.
- Comparar honestamente un modelo base contra uno optimizado y ajustar el **umbral de decisión** sin contaminar el conjunto de prueba.

### 📋 Instrucciones
- La carga de datos, el preprocesamiento (imputación, one-hot y escalamiento) y **todas las gráficas ya están programados**. No es el objetivo de hoy; ejecútalos sin modificarlos.
- Tu trabajo está en las celdas marcadas con **`✏️ TODO`** (son 6). Escribe tu código **solo** entre las marcas `▼▼▼ TU CÓDIGO AQUÍ ▼▼▼` y `▲▲▲ FIN DE TU CÓDIGO ▲▲▲`.
- Después de varios TODO hay una celda **✅ Verificación** que te dice si vas bien. Si marca error, léelo: te indica qué revisar.
- Las pistas están ocultas en los bloques **💡 Pista**; ábrelas solo si te atoras.
- Ejecuta las celdas **en orden** (en Colab: *Entorno de ejecución → Ejecutar todas* cuando termines, para comprobar que todo corre de principio a fin).

| TODO | Tema | Puntos |
|:---:|---|:---:|
| 1 | Función de métricas clínicas | 20 |
| 2 | Diccionario de hiperparámetros | 20 |
| 3 | Métrica a optimizar + justificación | 10 |
| 4 | Configurar y ejecutar `GridSearchCV` | 10 |
| 5 | Evaluar el modelo optimizado | 10 |
| 6 | Umbral de decisión | 10 |
| — | Preguntas de análisis (sección final) | 20 |
| | **Total** | **100** |

**Entrega:** sube a *Canvas* este notebook **ejecutado completo** (con todas las salidas visibles), con el nombre `Apellido_Nombre_Practica_Hiperparametros.ipynb`.

---
## 0 · Configuración del entorno *(ya resuelto — solo ejecuta)*

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score)

SEMILLA = 42
SKLEARN_1_8 = tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 8)
print(f"scikit-learn {sklearn.__version__}  |  pandas {pd.__version__}  |  numpy {np.__version__}")
print("API de regularización:", "l1_ratio (≥ 1.8)" if SKLEARN_1_8 else "penalty='elasticnet' + l1_ratio (< 1.8)")

scikit-learn 1.7.2  |  pandas 2.3.3  |  numpy 2.2.6
API de regularización: penalty='elasticnet' + l1_ratio (< 1.8)


In [2]:
# Herramientas de visualización (ejecuta sin modificar)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix

PALETA = {"navy": "#1E2664", "teal": "#0FA3B1", "amber": "#E8912B",
          "rojo": "#D1495B", "gris": "#6B7280", "fondo": "#F4F6FB"}
COLORES_MODELOS = ["#9AA3B5", "#0FA3B1", "#1E2664", "#E8912B"]

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 13, "font.size": 10.5})

def _aclarar(color, intensidad):
    """Mezcla un color con blanco. intensidad=1 -> color puro; 0 -> blanco."""
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(1 - intensidad * (1 - rgb))

def _caja(ax, x, y, w, h, color, radio=0.06, borde=None, lw=0, ls="-", z=1):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={radio}",
                                facecolor=color, edgecolor=borde or color, linewidth=lw,
                                linestyle=ls, zorder=z))

def graficar_matriz_confusion(y_real, y_pred, titulo="Matriz de confusión", ax=None,
                              clases=("Sin enfermedad", "Con enfermedad")):
    """Matriz de confusión clínica: conteos, % del total, significado clínico y métricas en los márgenes."""
    cm = confusion_matrix(y_real, y_pred, labels=[0, 1])
    vn, fp, fn, vp = cm.ravel()
    n, fila, col = cm.sum(), cm.sum(axis=1), cm.sum(axis=0)
    div = lambda a, b: a / b if b else 0.0

    L, G = 1.4, 0.08                      # lado de cada celda y separación
    W = 2 * L + G                         # ancho de la cuadrícula 2x2
    if ax is None:
        fig, ax = plt.subplots(figsize=(8.6, 7.3))
    ax.set_xlim(-1.0, W + 1.72)
    ax.set_ylim(-1.0, W + 1.0)
    ax.set_aspect("equal")
    ax.axis("off")

    celdas = {  # (fila_real, col_pred): (valor, sigla, nombre, significado, color)
        (0, 0): (vn, "VN", "Verdaderos negativos", "Sano bien descartado", PALETA["teal"]),
        (0, 1): (fp, "FP", "Falsos positivos", "Falsa alarma", PALETA["amber"]),
        (1, 0): (fn, "FN", "Falsos negativos", "¡Enfermo no detectado!", PALETA["rojo"]),
        (1, 1): (vp, "VP", "Verdaderos positivos", "Enfermo detectado", PALETA["navy"]),
    }
    for (i, j), (valor, sigla, nombre, significado, base) in celdas.items():
        x0, y0 = j * (L + G), (1 - i) * (L + G)
        inten = 0.25 + 0.75 * div(valor, fila[i])      # más intenso = mayor % de su fila
        txt = "white" if inten > 0.55 else PALETA["navy"]
        es_fn = (i, j) == (1, 0)
        _caja(ax, x0, y0, L, L, _aclarar(base, inten), radio=0.09,
              borde=PALETA["rojo"] if es_fn else None, lw=2.4 if es_fn else 0, ls="--" if es_fn else "-")
        ax.text(x0 + 0.11, y0 + L - 0.12, sigla, color=txt, fontsize=12, fontweight="bold", va="top")
        ax.text(x0 + L - 0.11, y0 + L - 0.13, f"{div(valor, n):.1%} del total", color=txt,
                fontsize=8.5, va="top", ha="right")
        ax.text(x0 + L / 2, y0 + 0.78, f"{valor}", color=txt, fontsize=36, fontweight="bold",
                ha="center", va="center")
        ax.text(x0 + L / 2, y0 + 0.40, nombre, color=txt, fontsize=9, ha="center", va="center",
                fontweight="bold")
        ax.text(x0 + L / 2, y0 + 0.20, significado, color=txt, fontsize=9, ha="center",
                va="center", style="italic")

    # Encabezados
    ax.text(W / 2, W + 0.45, "PREDICCIÓN DEL MODELO", ha="center", fontsize=10,
            color=PALETA["gris"], fontweight="bold")
    for j, c in enumerate(clases):
        ax.text(j * (L + G) + L / 2, W + 0.2, c, ha="center", va="center", fontsize=11,
                color=PALETA["navy"], fontweight="bold")
    ax.text(-0.82, W / 2, "DIAGNÓSTICO REAL", rotation=90, ha="center", va="center",
            fontsize=10, color=PALETA["gris"], fontweight="bold")
    for i, c in enumerate(clases):
        ax.text(-0.38, (1 - i) * (L + G) + L / 2, c.replace(" ", "\n", 1), rotation=90,
                ha="center", va="center", fontsize=11, color=PALETA["navy"], fontweight="bold")

    # Métricas por fila (derecha) y por columna (abajo)
    xm, wm, hm = W + 0.18, 1.5, 0.8
    fila_info = [("Especificidad", div(vn, fila[0]), f"{vn} de {fila[0]} sanos", PALETA["teal"]),
                 ("Sensibilidad", div(vp, fila[1]), f"{vp} de {fila[1]} enfermos", PALETA["navy"])]
    for i, (nom, val, det, c) in enumerate(fila_info):
        y0 = (1 - i) * (L + G) + (L - hm) / 2
        _caja(ax, xm, y0, wm, hm, _aclarar(c, 0.12), borde=c, lw=1.5, radio=0.12)
        ax.text(xm + wm / 2, y0 + 0.62, nom, ha="center", va="center", fontsize=9.5, color=c, fontweight="bold")
        ax.text(xm + wm / 2, y0 + 0.37, f"{val:.1%}", ha="center", va="center", fontsize=17, color=c, fontweight="bold")
        ax.text(xm + wm / 2, y0 + 0.12, det, ha="center", va="center", fontsize=8, color=PALETA["gris"])

    col_info = [("VPN", div(vn, col[0]), f"{vn} de {col[0]} negativos", PALETA["teal"]),
                ("Precisión (VPP)", div(vp, col[1]), f"{vp} de {col[1]} positivos", PALETA["navy"])]
    yb = -0.12 - hm
    for j, (nom, val, det, c) in enumerate(col_info):
        x0 = j * (L + G) + (L - wm + 0.2) / 2
        w = wm - 0.2
        _caja(ax, x0, yb, w, hm, _aclarar(c, 0.12), borde=c, lw=1.5, radio=0.12)
        ax.text(x0 + w / 2, yb + 0.62, nom, ha="center", va="center", fontsize=9.5, color=c, fontweight="bold")
        ax.text(x0 + w / 2, yb + 0.37, f"{val:.1%}", ha="center", va="center", fontsize=17, color=c, fontweight="bold")
        ax.text(x0 + w / 2, yb + 0.12, det, ha="center", va="center", fontsize=8, color=PALETA["gris"])

    exact, f1 = div(vp + vn, n), div(2 * vp, 2 * vp + fp + fn)
    _caja(ax, xm, yb, wm, hm, PALETA["navy"], radio=0.12)
    ax.text(xm + wm / 2, yb + 0.55, f"Exactitud  {exact:.1%}", ha="center", va="center",
            color="white", fontsize=10, fontweight="bold")
    ax.text(xm + wm / 2, yb + 0.24, f"F1  {f1:.3f}", ha="center", va="center", color="white",
            fontsize=14, fontweight="bold")

    ax.text(W / 2 + 0.4, W + 1.0, titulo, ha="center", va="top", fontsize=14.5,
            fontweight="bold", color=PALETA["navy"])
    ax.text(W / 2 + 0.4, W + 0.76, f"n = {n} pacientes", ha="center", va="top", fontsize=9,
            color=PALETA["gris"])
    return ax

def graficar_matrices_lado_a_lado(casos):
    """casos: lista de tuplas (y_real, y_pred, titulo)."""
    fig, axes = plt.subplots(1, len(casos), figsize=(8.2 * len(casos), 7.4))
    axes = np.atleast_1d(axes)
    for ax, (yr, yp, t) in zip(axes, casos):
        graficar_matriz_confusion(yr, yp, titulo=t, ax=ax)
    plt.tight_layout()
    plt.show()

ETIQUETAS_METRICAS = {"exactitud": "Exactitud", "sensibilidad": "Sensibilidad",
                      "especificidad": "Especificidad", "precision": "Precisión (VPP)",
                      "f1": "F1", "auc": "AUC-ROC"}

def mostrar_metricas(m, titulo="Métricas"):
    """Imprime el diccionario de métricas de forma legible."""
    print(f"── {titulo} " + "─" * max(0, 44 - len(titulo)))
    print(f"   VP={m['VP']}  FN={m['FN']}  VN={m['VN']}  FP={m['FP']}")
    for k, nombre in ETIQUETAS_METRICAS.items():
        v = m[k]
        barra = "█" * int(round((v or 0) * 20))
        print(f"   {nombre:<16} {v:6.3f}  {barra}")

def tabla_comparativa(resultados):
    """resultados: dict {nombre_modelo: dict_de_metricas} -> DataFrame comparativo."""
    filas = {"Enfermos no detectados (FN)": "FN", "Falsas alarmas (FP)": "FP"}
    datos = {}
    for nombre, m in resultados.items():
        col = {ETIQUETAS_METRICAS[k]: round(float(m[k]), 3) for k in ETIQUETAS_METRICAS}
        col.update({et: int(m[k]) for et, k in filas.items()})
        datos[nombre] = pd.Series(col, dtype=object)
    return pd.DataFrame(datos)

def graficar_comparacion_metricas(resultados, titulo="Comparación de modelos en el conjunto de prueba"):
    """Gráfica de puntos: cada fila es una métrica y cada color un modelo."""
    claves = list(ETIQUETAS_METRICAS)
    fig, ax = plt.subplots(figsize=(10, 0.62 * len(claves) + 1.8))
    ypos = np.arange(len(claves))[::-1]
    todos = np.array([[float(m[k]) for k in claves] for m in resultados.values()])
    for y in ypos[:-1]:
        ax.axhline(y - 0.5, color="#E3E6EE", lw=1, zorder=0)
    n = len(resultados)
    desplaz = np.linspace(0.2, -0.2, n) if n > 1 else [0]
    for k, (nombre, m) in enumerate(resultados.items()):
        vals = [float(m[c]) for c in claves]
        ax.scatter(vals, ypos + desplaz[k], s=150, color=COLORES_MODELOS[k % 4], label=nombre,
                   zorder=3, edgecolor="white", linewidth=1.5)
        for v, y in zip(vals, ypos + desplaz[k]):
            ax.text(v + 0.004, y, f"{v:.3f}", va="center", fontsize=8.5, color=COLORES_MODELOS[k % 4],
                    fontweight="bold")
    ax.set_yticks(ypos)
    ax.set_yticklabels([ETIQUETAS_METRICAS[c] for c in claves], fontsize=11)
    ax.set_xlim(max(0, todos.min() - 0.04), min(1.0, todos.max()) + 0.035)
    ax.grid(axis="x", alpha=0.3)
    ax.tick_params(axis="y", length=0)
    ax.set_title(titulo, color=PALETA["navy"], loc="left")
    ax.set_xlabel("Valor de la métrica  (eje recortado para apreciar diferencias pequeñas)",
                  color=PALETA["gris"])
    ax.set_ylim(-0.5, len(claves) - 0.5)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, title="Modelo")
    plt.tight_layout()
    plt.show()

def graficar_busqueda(busqueda):
    """Puntaje de validación cruzada vs C para cada combinación del resto de hiperparámetros."""
    res = pd.DataFrame(busqueda.cv_results_).copy()
    col_c = "param_modelo__C"
    if col_c not in res.columns:
        print("⚠️ Tu param_grid no incluye 'modelo__C'; no se puede dibujar esta gráfica.")
        return
    otras = [c for c in res.columns if c.startswith("param_") and c != col_c]
    for c in otras:   # texto legible (None → "None") para agrupar y rotular
        res[c] = res[c].map(lambda v: "None" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
    res[col_c] = res[col_c].astype(float)
    col_panel = "param_modelo__class_weight" if "param_modelo__class_weight" in otras else None
    col_lineas = [c for c in otras if c != col_panel]
    paneles = list(res[col_panel].unique()) if col_panel else [None]

    mejor = res.loc[busqueda.best_index_]
    piso = mejor["mean_test_score"] - mejor["std_test_score"]
    colores = [PALETA["navy"], PALETA["teal"], PALETA["amber"], PALETA["rojo"], PALETA["gris"],
               "#7B61FF", "#2E8B57"]
    fig, axes = plt.subplots(1, len(paneles), figsize=(6.6 * len(paneles), 4.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, paneles):
        sub = res if p is None else res[res[col_panel] == p]
        grupos = sub.groupby(col_lineas) if col_lineas else [((), sub)]
        for k, (clave, g) in enumerate(grupos):
            g = g.sort_values(col_c)
            clave = clave if isinstance(clave, tuple) else (clave,)
            etiqueta = ", ".join(f"{c.replace('param_modelo__', '')} = {v}"
                                 for c, v in zip(col_lineas, clave)) or "C"
            ax.plot(g[col_c], g["mean_test_score"], marker="o", lw=2.2, ms=6,
                    color=colores[k % len(colores)], label=etiqueta)
        ax.axhspan(piso, mejor["mean_test_score"], color=PALETA["teal"], alpha=0.10,
                   label="Zona de empate (mejor − 1 desv. est.)")
        ax.axhline(piso, color=PALETA["teal"], ls="--", lw=1)
        if p is None or mejor[col_panel] == p:
            ax.scatter([mejor[col_c]], [mejor["mean_test_score"]], marker="*", s=520,
                       color="#FFD23F", edgecolor=PALETA["navy"], linewidth=1.6, zorder=5,
                       label="Mejor combinación")
        ax.set_xscale("log")
        ax.set_xlabel("C  (← más regularización  |  menos regularización →)")
        ax.set_title("" if p is None else f"class_weight = {p}", color=PALETA["navy"])
        ax.grid(alpha=0.3)
    y_inf = max(0, mejor["mean_test_score"] - 0.10)          # enfocamos la zona útil
    axes[0].set_ylim(y_inf, res["mean_test_score"].max() + 0.012)
    if (res["mean_test_score"] < y_inf).any():
        for ax in axes:
            ax.text(0.02, 0.03, "↓ líneas que salen por abajo: el modelo colapsa\n   por exceso de regularización (C muy pequeño)",
                    transform=ax.transAxes, fontsize=8.5, color=PALETA["gris"], va="bottom",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.9))
    axes[0].set_ylabel(f"{busqueda.scoring} promedio (VC 5 pliegues)")
    axes[-1].legend(loc="lower right", fontsize=8.5, framealpha=0.95)
    fig.suptitle("Resultados de la búsqueda en rejilla", fontweight="bold", color=PALETA["navy"], x=0.01,
                 ha="left", fontsize=14)
    plt.tight_layout()
    plt.show()

def graficar_coeficientes(modelos):
    """modelos: dict {nombre: pipeline entrenado}. Compara los coeficientes de la regresión logística."""
    datos = {}
    for nombre, pipe in modelos.items():
        nombres = pipe.named_steps["prep"].get_feature_names_out()
        nombres = [n.split("__", 1)[1] for n in nombres]
        datos[nombre] = pd.Series(pipe.named_steps["modelo"].coef_[0], index=nombres)
    df_c = pd.DataFrame(datos)
    df_c = df_c.reindex(df_c.abs().max(axis=1).sort_values().index)
    fig, ax = plt.subplots(figsize=(10, 0.36 * len(df_c) + 1.6))
    y = np.arange(len(df_c))
    alto = 0.8 / len(modelos)
    for k, nombre in enumerate(df_c.columns):
        pos = y - 0.4 + alto * (k + 0.5)
        ax.barh(pos, df_c[nombre], height=alto * 0.92, color=COLORES_MODELOS[k % 4], label=nombre)
        for yy, v in zip(pos, df_c[nombre]):
            if v == 0:
                ax.text(0.02, yy, "0  (variable eliminada)", va="center", fontsize=8,
                        color=PALETA["rojo"], fontweight="bold")
    ax.axvline(0, color=PALETA["navy"], lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(df_c.index, fontsize=9.5)
    ax.set_xlabel("Coeficiente  (← protege  |  aumenta el riesgo predicho →)")
    ax.set_title("Coeficientes de la regresión logística: efecto de la regularización",
                 color=PALETA["navy"], loc="left")
    ax.grid(axis="x", alpha=0.3)
    ax.legend(loc="lower right", frameon=True)
    plt.tight_layout()
    plt.show()
    return df_c

def graficar_umbral(umbrales, sens, espec, umbral_elegido=None, sens_minima=None):
    """Sensibilidad y especificidad (validación cruzada) en función del umbral de decisión."""
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(umbrales, sens, color=PALETA["navy"], lw=2.6, label="Sensibilidad")
    ax.plot(umbrales, espec, color=PALETA["teal"], lw=2.6, label="Especificidad")
    ax.axvline(0.5, color=PALETA["gris"], ls=":", lw=1.5)
    ax.text(0.505, 0.03, "umbral por\ndefecto 0.50", color=PALETA["gris"], fontsize=8.5)
    if sens_minima is not None:
        ax.axhline(sens_minima, color=PALETA["navy"], ls="--", lw=1)
        ax.text(umbrales[0], sens_minima + 0.012, f"sensibilidad mínima = {sens_minima:.2f}",
                color=PALETA["navy"], fontsize=8.5)
    if umbral_elegido is not None:
        i = int(np.argmin(np.abs(np.asarray(umbrales) - umbral_elegido)))
        ax.axvline(umbral_elegido, color=PALETA["amber"], lw=2.2)
        ax.scatter([umbral_elegido] * 2, [sens[i], espec[i]], s=90, color=PALETA["amber"],
                   edgecolor=PALETA["navy"], zorder=5)
        ax.text(umbral_elegido - 0.01, 0.12, f"umbral elegido\n{umbral_elegido:.2f}", ha="right",
                color=PALETA["amber"], fontweight="bold", fontsize=9.5)
    ax.set_xlabel("Umbral de decisión  (probabilidad a partir de la cual se predice «con enfermedad»)")
    ax.set_ylabel("Valor (validación cruzada)")
    ax.set_ylim(0, 1.03)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
    ax.set_title("Compromiso sensibilidad–especificidad según el umbral", color=PALETA["navy"], loc="left")
    plt.tight_layout()
    plt.show()

print("✅ Funciones de visualización cargadas.")

✅ Funciones de visualización cargadas.


---
## 1 · Datos y preprocesamiento *(ya resuelto — tema de la Unidad 4)*

Usamos el **Heart Failure Prediction Dataset** (fedesoriano, Kaggle): 918 pacientes de cinco cohortes cardiológicas y 11 variables clínicas.

| Variable | Significado clínico |
|---|---|
| `Age` | Edad (años) |
| `Sex` | Sexo (M/F) |
| `ChestPainType` | Tipo de dolor torácico: TA angina típica, ATA atípica, NAP no anginoso, ASY asintomático |
| `RestingBP` | Presión arterial sistólica en reposo (mm Hg) |
| `Cholesterol` | Colesterol sérico (mg/dL) |
| `FastingBS` | Glucosa en ayuno > 120 mg/dL (1 = sí) |
| `RestingECG` | ECG en reposo: Normal, ST (anomalía ST-T), LVH (hipertrofia ventricular izquierda) |
| `MaxHR` | Frecuencia cardiaca máxima alcanzada en prueba de esfuerzo |
| `ExerciseAngina` | Angina inducida por ejercicio (Y/N) |
| `Oldpeak` | Depresión del segmento ST con el esfuerzo (mm) |
| `ST_Slope` | Pendiente del segmento ST en el pico de esfuerzo: Up, Flat, Down |
| **`HeartDisease`** | **Variable objetivo: 1 = con enfermedad cardiaca, 0 = sin enfermedad** |

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                            "fedesoriano/heart-failure-prediction", "heart.csv")

print(df["HeartDisease"].map({0: "Sin enfermedad (0)", 1: "Con enfermedad (1)"}).value_counts().to_string())
df.head()

c:\Users\chels\Documents\GitHub\IA_practicas\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HeartDisease
Con enfermedad (1)    508
Sin enfermedad (0)    410


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
# ── Preprocesamiento: todo vive DENTRO del pipeline para evitar fuga de información ──
df_modelo = df.copy()
# Un colesterol o una presión arterial de 0 son fisiológicamente imposibles → valores faltantes
df_modelo["Cholesterol"] = df_modelo["Cholesterol"].replace(0, np.nan)
df_modelo["RestingBP"] = df_modelo["RestingBP"].replace(0, np.nan)

X = df_modelo.drop(columns="HeartDisease")
y = df_modelo["HeartDisease"]

VARS_NUM = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
VARS_CAT = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
VARS_BIN = ["FastingBS"]

# Partición 80/20 estratificada: el conjunto de prueba NO se toca hasta evaluar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA)

preprocesador = ColumnTransformer([
    ("num", Pipeline([("imputar", SimpleImputer(strategy="median")),
                      ("escalar", StandardScaler())]), VARS_NUM),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), VARS_CAT),
    ("bin", "passthrough", VARS_BIN),
])

def crear_pipeline():
    # Devuelve un pipeline NUEVO: preprocesamiento + regresión logística con valores por defecto.
    # solver="saga" es el único que acepta cualquier mezcla L1/L2 (l1_ratio entre 0 y 1).
    extra = {} if SKLEARN_1_8 else {"penalty": "elasticnet"}   # compatibilidad con versiones < 1.8
    modelo = LogisticRegression(solver="saga", C=1.0, l1_ratio=0.0, class_weight=None,
                                max_iter=5000, random_state=SEMILLA, **extra)
    return Pipeline([("prep", clone(preprocesador)), ("modelo", modelo)])

# Validación cruzada estratificada de 5 pliegues (se usa en toda la práctica)
validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

print(f"Entrenamiento: {len(X_train)} pacientes  ({y_train.mean():.1%} con enfermedad)")
print(f"Prueba:        {len(X_test)} pacientes  ({y_test.mean():.1%} con enfermedad)")
print("\nPasos del pipeline:", [nombre for nombre, _ in crear_pipeline().steps])

Entrenamiento: 734 pacientes  (55.3% con enfermedad)
Prueba:        184 pacientes  (55.4% con enfermedad)

Pasos del pipeline: ['prep', 'modelo']


---
## 2 · Métricas clínicas

La **exactitud** sola engaña en medicina: no distingue entre "mandar a casa a un enfermo" y "pedir un estudio de más a un sano". Por eso usamos métricas que separan ambos errores. Piensa siempre en **frecuencias naturales**:

| Métrica | Pregunta clínica que responde | Fórmula con la matriz | Función de `sklearn` |
|---|---|---|---|
| **Sensibilidad** (recall) | De cada 100 **enfermos**, ¿a cuántos detecta? | VP / (VP + FN) | `recall_score` |
| **Especificidad** | De cada 100 **sanos**, ¿a cuántos descarta correctamente? | VN / (VN + FP) | `recall_score` de la clase 0 |
| **Precisión** (VPP) | De cada 100 pacientes que el modelo marca como enfermos, ¿cuántos lo están? | VP / (VP + FP) | `precision_score` |
| **F1** | Equilibrio entre precisión y sensibilidad (un solo número) | 2·VP / (2·VP + FP + FN) | `f1_score` |
| **Exactitud** | De cada 100 pacientes, ¿cuántos clasifica bien? | (VP + VN) / total | `accuracy_score` |
| **AUC-ROC** | Si tomo un enfermo y un sano al azar, ¿qué tan seguido el enfermo recibe mayor probabilidad? | — (usa probabilidades) | `roc_auc_score` |

> En `confusion_matrix` de scikit-learn, las filas son la **realidad** y las columnas la **predicción**, en el orden `[0, 1]`. Por eso `.ravel()` devuelve `VN, FP, FN, VP` en ese orden.

### ✏️ TODO 1 — Completa la función `calcular_metricas` *(20 pts)*
Usa **exclusivamente funciones de `sklearn.metrics`** (ya están importadas). No calcules las fórmulas a mano.

<details><summary>💡 Pista 1 — matriz de confusión</summary>

`confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()` devuelve 4 números. Asígnalos a `vn, fp, fn, vp` en una sola línea.
</details>

<details><summary>💡 Pista 2 — especificidad</summary>

La especificidad es la sensibilidad **de la clase negativa**. `recall_score` tiene un parámetro `pos_label` que indica cuál clase se considera "positiva".
</details>

<details><summary>💡 Pista 3 — AUC</summary>

El AUC evalúa qué tan bien **ordena** el modelo a los pacientes, así que necesita `y_proba` (probabilidades), no `y_pred` (ceros y unos).
</details>

In [ ]:
# ✏️ TODO 1 ─────────────────────────────────────────────────────────────
def calcular_metricas(y_real, y_pred, y_proba):
    # Devuelve un diccionario con la matriz de confusión desglosada y las métricas clínicas.
    #   y_real  : etiquetas verdaderas (0/1)
    #   y_pred  : etiquetas predichas por el modelo (0/1)
    #   y_proba : probabilidad predicha de la clase 1 (con enfermedad)

    # ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼
    vn, fp, fn, vp = confusion_matrix(y_real, y_pred, labels=[0,1]).ravel()  # desempaqueta la matriz de confusión
    exactitud     = None
    sensibilidad  = None
    especificidad = None
    precision     = None
    f1            = None
    auc           = None
    # ▲▲▲ FIN DE TU CÓDIGO ▲▲▲

    return {"VN": vn, "FP": fp, "FN": fn, "VP": vp,
            "exactitud": exactitud, "sensibilidad": sensibilidad,
            "especificidad": especificidad, "precision": precision,
            "f1": f1, "auc": auc}

In [ ]:
# ✅ VERIFICACIÓN TODO 1 — caso de prueba con 12 pacientes (7 enfermos, 5 sanos)
_y_real  = np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
_y_pred  = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1])
_y_proba = np.array([.95, .90, .85, .80, .70, .60, .30, .10, .20, .45, .65, .55])

_esperado = {"VN": 3, "FP": 2, "FN": 1, "VP": 6, "exactitud": 0.75, "sensibilidad": 6 / 7,
             "especificidad": 0.60, "precision": 0.75, "f1": 0.80, "auc": 31 / 35}
_pistas = {"sensibilidad": "usa recall_score(y_real, y_pred)",
           "especificidad": "es el recall de la clase 0 → revisa el parámetro pos_label",
           "precision": "usa precision_score", "f1": "usa f1_score",
           "auc": "roc_auc_score necesita y_proba, no y_pred",
           "exactitud": "usa accuracy_score"}

_m = calcular_metricas(_y_real, _y_pred, _y_proba)
_errores = 0
for clave, valor_esperado in _esperado.items():
    valor = _m.get(clave)
    if valor is None:
        print(f"⏳ {clave:<14} pendiente")
        _errores += 1
    elif not np.isclose(float(valor), valor_esperado):
        print(f"❌ {clave:<14} obtuviste {float(valor):.3f}, se esperaba {valor_esperado:.3f}  → {_pistas.get(clave, 'revisa el orden de .ravel(): VN, FP, FN, VP')}")
        _errores += 1
    else:
        print(f"✅ {clave:<14} {float(valor):.3f}")
assert _errores == 0, f"Hay {_errores} métrica(s) por corregir en calcular_metricas."
print("\n🎉 ¡Tu función de métricas es correcta!")

---
## 3 · Modelo base: regresión logística **sin optimizar** *(ya resuelto)*

Entrenamos el pipeline con los valores por defecto (`C = 1`, penalización L2, sin pesos de clase) y lo evaluamos **una sola vez** en prueba. Esta es la referencia que la búsqueda de hiperparámetros tiene que superar.

In [ ]:
modelo_base = crear_pipeline()
modelo_base.fit(X_train, y_train)

y_pred_base  = modelo_base.predict(X_test)
y_proba_base = modelo_base.predict_proba(X_test)[:, 1]
metricas_base = calcular_metricas(y_test, y_pred_base, y_proba_base)

mostrar_metricas(metricas_base, "Modelo base (prueba)")
graficar_matriz_confusion(y_test, y_pred_base, "Modelo base · C = 1, sin optimizar")
plt.show()

> 🩺 **Lee la matriz como clínico:** la celda roja punteada (**FN**) son pacientes con enfermedad cardiaca que el modelo mandaría a casa. Las métricas en los márgenes se leen por **fila** (sensibilidad, especificidad) y por **columna** (VPN, precisión).

---
## 4 · El diccionario de hiperparámetros

Un **hiperparámetro** no se aprende de los datos: lo decides tú **antes** de entrenar. En `LogisticRegression` los tres más importantes son:

| Hiperparámetro | Qué controla | Intuición clínica |
|---|---|---|
| **`C`** | Inverso de la fuerza de regularización. `C` pequeño = modelo más "simple" y prudente; `C` grande = el modelo se ajusta más a los datos de entrenamiento. | Un médico que se deja llevar por cada detalle raro de un expediente (C grande) vs. uno que solo confía en los signos más sólidos (C pequeño). |
| **`l1_ratio`** | Tipo de regularización: `0` = **L2 (Ridge)** encoge todos los coeficientes; `1` = **L1 (Lasso)** puede llevar coeficientes a **cero exacto** (elimina variables); valores intermedios = **Elastic-Net** (mezcla). | L1 funciona como un filtro que descarta estudios clínicos que no aportan. |
| **`class_weight`** | `None` = todos los pacientes pesan igual; `"balanced"` = compensa si una clase es menos frecuente. | Útil cuando hay pocos enfermos; aquí las clases están casi balanceadas (55 % / 45 %), así que la búsqueda dirá si vale la pena. |

> ⚙️ **Nota de versión:** desde scikit-learn 1.8 el tipo de penalización se controla solo con `l1_ratio` (el parámetro `penalty` quedó obsoleto). El `crear_pipeline()` de arriba ya contempla ambas versiones.

### 🔗 El prefijo `modelo__`
Como la regresión logística vive **dentro** de un `Pipeline`, `GridSearchCV` necesita saber a qué paso pertenece cada hiperparámetro. La regla es **`<nombre_del_paso>__<hiperparámetro>`** (dos guiones bajos). Nuestro paso se llama `"modelo"`, así que `C` se escribe `"modelo__C"`.

### ✏️ TODO 2 — Define desde cero el diccionario `param_grid` *(20 pts)*
Requisitos mínimos:
1. **`C`**: al menos **6 valores** en escala logarítmica que vayan **de 0.001 a 100** (inclusive).
2. **`l1_ratio`**: al menos los valores **0, 0.5 y 1**.
3. **`class_weight`**: las opciones **`None`** y **`"balanced"`**.

<details><summary>💡 Pista 1 — estructura</summary>

Es un diccionario `{ "nombre_del_hiperparámetro": [lista de valores a probar], ... }`. Cada clave lleva el prefijo `modelo__`.
</details>

<details><summary>💡 Pista 2 — escala logarítmica</summary>

`np.logspace(a, b, n)` genera `n` valores entre `10**a` y `10**b`. ¿Qué exponentes corresponden a 0.001 y a 100?
</details>

In [ ]:
# ✏️ TODO 2 ─────────────────────────────────────────────────────────────
# ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼
param_grid = None
# ▲▲▲ FIN DE TU CÓDIGO ▲▲▲

In [ ]:
# ✅ VERIFICACIÓN TODO 2
assert isinstance(param_grid, dict), "param_grid debe ser un diccionario {'modelo__hiperparámetro': [valores]}."
_sin_prefijo = [k for k in param_grid if not k.startswith("modelo__")]
assert not _sin_prefijo, f"Estas claves no llevan el prefijo 'modelo__': {_sin_prefijo}"
_faltan = [k for k in ["modelo__C", "modelo__l1_ratio", "modelo__class_weight"] if k not in param_grid]
assert not _faltan, f"Faltan estos hiperparámetros: {_faltan}"

_C = np.asarray(param_grid["modelo__C"], dtype=float)
assert len(_C) >= 6, f"Incluye al menos 6 valores de C (tienes {len(_C)})."
assert _C.min() <= 0.0011 and _C.max() >= 99.9, f"C debe cubrir de 0.001 a 100 (tu rango: {_C.min():g} – {_C.max():g})."
_l1 = {float(v) for v in param_grid["modelo__l1_ratio"]}
assert {0.0, 0.5, 1.0} <= _l1, f"l1_ratio debe incluir 0, 0.5 y 1 (tienes {sorted(_l1)})."
_cw = list(param_grid["modelo__class_weight"])
assert None in _cw and "balanced" in _cw, "class_weight debe incluir None y 'balanced'."
crear_pipeline().set_params(**{k: list(v)[0] for k, v in param_grid.items()})   # ¿nombres válidos?

_n_comb = int(np.prod([len(list(v)) for v in param_grid.values()]))
print("✅ Diccionario válido")
for k, v in param_grid.items():
    print(f"   {k:<22} {len(list(v)):>2} valores: {[round(float(x), 4) if isinstance(x, (int, float, np.number)) and not isinstance(x, bool) else x for x in list(v)]}")
print(f"\n   {_n_comb} combinaciones × {validacion.get_n_splits()} pliegues = {_n_comb * validacion.get_n_splits()} entrenamientos")

---
## 5 · Búsqueda en rejilla con validación cruzada

`GridSearchCV` entrena el pipeline **una vez por cada combinación y por cada pliegue**, y se queda con la combinación de mejor **puntaje promedio en validación**. Al final re-entrena automáticamente el ganador con todo el conjunto de entrenamiento (`refit=True`).

La pregunta clave es **¿qué significa "mejor"?** Eso lo decide el parámetro `scoring`:

| `scoring` | Optimiza… | ¿Riesgo? |
|---|---|---|
| `"accuracy"` | aciertos totales | ignora qué tipo de error se comete |
| `"recall"` | solo sensibilidad | 🤔 lo veremos en un experimento más adelante… |
| `"f1"` | equilibrio precisión–sensibilidad | no considera los verdaderos negativos |
| `"roc_auc"` | capacidad de ordenar enfermos sobre sanos | no depende del umbral 0.5 |
| `"balanced_accuracy"` | promedio de sensibilidad y especificidad | trata ambos errores como igual de graves |

### ✏️ TODO 3 — Elige la métrica a optimizar *(10 pts)*
Asigna a `metrica_busqueda` **una** de las opciones de la tabla y justifica tu elección en la celda de texto de abajo (2–3 líneas, pensando en el uso clínico del modelo).

### ✏️ TODO 4 — Configura y ejecuta `GridSearchCV` *(10 pts)*
Crea el objeto `busqueda` con: el pipeline de `crear_pipeline()`, tu `param_grid`, tu métrica, la validación cruzada `validacion` y `n_jobs=-1` (usa todos los núcleos). Después **entrénalo** con los datos de entrenamiento.

<details><summary>💡 Pista</summary>

`GridSearchCV(estimator=..., param_grid=..., scoring=..., cv=..., n_jobs=-1)` y luego `.fit(X_train, y_train)`. ¡Nunca con `X_test`!
</details>

In [ ]:
# ✏️ TODO 3 y 4 ─────────────────────────────────────────────────────────
# ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼
metrica_busqueda = None      # "accuracy", "recall", "f1", "roc_auc" o "balanced_accuracy"
busqueda = None              # GridSearchCV(...)

# entrena la búsqueda aquí

# ▲▲▲ FIN DE TU CÓDIGO ▲▲▲

**✍️ Justificación de la métrica elegida (TODO 3):**

*Escribe aquí tu respuesta…*

In [ ]:
# ✅ VERIFICACIÓN TODO 3 y 4 + resumen de la búsqueda
assert metrica_busqueda in ["accuracy", "recall", "f1", "roc_auc", "balanced_accuracy"], \
    "metrica_busqueda debe ser una de las cinco opciones de la tabla."
assert isinstance(busqueda, GridSearchCV), "busqueda debe ser un objeto GridSearchCV."
assert hasattr(busqueda, "best_estimator_"), "Falta entrenar la búsqueda con .fit(X_train, y_train)."
assert busqueda.n_splits_ == 5, "Usa cv=validacion (5 pliegues estratificados)."

# Comparación JUSTA: el modelo base evaluado con la misma métrica y los mismos pliegues
_cv_base = cross_val_score(crear_pipeline(), X_train, y_train, cv=validacion, scoring=metrica_busqueda)
_std_mejor = busqueda.cv_results_["std_test_score"][busqueda.best_index_]
print(f"Métrica optimizada: {metrica_busqueda}\n")
print(f"   Modelo base (C=1, L2)     {_cv_base.mean():.4f} ± {_cv_base.std():.4f}")
print(f"   Mejor de la búsqueda      {busqueda.best_score_:.4f} ± {_std_mejor:.4f}")
print("\nMejores hiperparámetros:")
for k, v in busqueda.best_params_.items():
    print(f"   {k.replace('modelo__', ''):<14} {round(v, 4) if isinstance(v, float) else v}")

In [ ]:
# Las 10 mejores combinaciones (ranking por puntaje promedio de validación)
_res = pd.DataFrame(busqueda.cv_results_)
_cols = [c for c in _res.columns if c.startswith("param_")]
top10 = (_res.sort_values("rank_test_score")
             [["rank_test_score"] + _cols + ["mean_test_score", "std_test_score"]]
             .rename(columns=lambda c: c.replace("param_modelo__", ""))
             .head(10).reset_index(drop=True))
top10.style.format({"C": "{:.4g}", "mean_test_score": "{:.4f}", "std_test_score": "{:.4f}"}) \
     .background_gradient(subset=["mean_test_score"], cmap="GnBu")

In [ ]:
graficar_busqueda(busqueda)

> 📈 **Cómo leer la gráfica:** cada línea es una combinación de `l1_ratio`; el eje X es `C` en escala logarítmica. La ⭐ es el ganador. La **franja turquesa** marca la *zona de empate*: toda configuración que caiga dentro está a menos de una desviación estándar del ganador, así que **no podemos afirmar que sea peor**. Observa también qué pasa a la izquierda (C muy pequeño): el modelo se "apaga" por exceso de regularización.

---
## 6 · Evaluación del modelo optimizado en el conjunto de prueba

### ✏️ TODO 5 — Evalúa el ganador de la búsqueda *(10 pts)*
1. Obtén el **pipeline ganador** ya re-entrenado a partir de `busqueda`.
2. Calcula sus predicciones (`y_pred_opt`) y sus probabilidades de la clase 1 (`y_proba_opt`) en **prueba**.
3. Calcula sus métricas con **tu** función `calcular_metricas`.

<details><summary>💡 Pista</summary>

El atributo que termina en guion bajo y contiene al mejor modelo es `best_estimator_`. Para las probabilidades de la clase 1 recuerda la columna `[:, 1]` de `predict_proba`, igual que en el modelo base.
</details>

In [ ]:
# ✏️ TODO 5 ─────────────────────────────────────────────────────────────
# ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼
mejor_modelo  = None
y_pred_opt    = None
y_proba_opt   = None
metricas_opt  = None
# ▲▲▲ FIN DE TU CÓDIGO ▲▲▲

In [ ]:
# ✅ VERIFICACIÓN TODO 5
assert mejor_modelo is busqueda.best_estimator_, "mejor_modelo debe ser busqueda.best_estimator_ (no lo re-entrenes a mano)."
assert y_pred_opt is not None and len(y_pred_opt) == len(y_test), "Calcula y_pred_opt sobre X_test."
assert y_proba_opt is not None and np.ndim(y_proba_opt) == 1 and np.all((y_proba_opt >= 0) & (y_proba_opt <= 1)), \
    "y_proba_opt debe ser un vector de probabilidades de la clase 1 → predict_proba(X_test)[:, 1]"
assert metricas_opt is not None, "Calcula metricas_opt con calcular_metricas."
mostrar_metricas(metricas_opt, "Modelo optimizado (prueba)")

In [ ]:
graficar_matrices_lado_a_lado([
    (y_test, y_pred_base, "Modelo base · C = 1"),
    (y_test, y_pred_opt,  "Modelo optimizado · GridSearchCV"),
])

In [ ]:
resultados = {"Base": metricas_base, "Optimizado": metricas_opt}
display(tabla_comparativa(resultados))
graficar_comparacion_metricas(resultados)

### 🔬 ¿Qué cambió dentro del modelo?
La regularización no solo cambia las métricas: cambia **qué variables usa el modelo**. Si tu ganador tiene `l1_ratio > 0`, algunas variables pueden quedar con coeficiente **cero exacto** (el modelo las descartó).

In [ ]:
coeficientes = graficar_coeficientes({"Base": modelo_base, "Optimizado": mejor_modelo})
eliminadas = coeficientes.index[coeficientes["Optimizado"] == 0].tolist()
print("Variables eliminadas por el modelo optimizado:", eliminadas if eliminadas else "ninguna")

---
## 🧪 Experimento guiado: la trampa de optimizar solo la sensibilidad *(ya resuelto — ejecuta y observa)*

"En medicina lo más importante es no dejar ir a un enfermo… ¡entonces optimicemos solo la sensibilidad!" Veamos qué pasa si le pedimos a `GridSearchCV` exactamente eso.

In [ ]:
busqueda_trampa = GridSearchCV(
    crear_pipeline(),
    {"modelo__C": [0.001, 0.01, 0.1, 1, 10, 100],
     "modelo__l1_ratio": [0.0, 1.0],
     "modelo__class_weight": [None, "balanced"]},
    scoring="recall", cv=validacion, n_jobs=-1,
).fit(X_train, y_train)

print("Ganador al optimizar SOLO sensibilidad:", {k.replace("modelo__", ""): v for k, v in busqueda_trampa.best_params_.items()})
print(f"Sensibilidad promedio en validación: {busqueda_trampa.best_score_:.3f}")

y_pred_trampa  = busqueda_trampa.predict(X_test)
metricas_trampa = calcular_metricas(y_test, y_pred_trampa, busqueda_trampa.predict_proba(X_test)[:, 1])
graficar_matriz_confusion(y_test, y_pred_trampa, "La trampa: optimizar solo sensibilidad")
plt.show()

> 🤔 **Piénsalo:** ¿de qué le serviría este modelo al servicio de cardiología? ¿Qué métrica(s) lo delatan de inmediato? (Lo responderás en las preguntas finales.)

---
## 7 · Ajuste del umbral de decisión

Por defecto, `predict()` dice "con enfermedad" cuando la probabilidad es **≥ 0.50**. Pero ese 0.50 **también es una decisión**: si bajamos el umbral, detectamos más enfermos (↑ sensibilidad) a costa de más falsas alarmas (↓ especificidad).

⚠️ **Regla de oro:** el umbral se elige con datos de **entrenamiento** (mediante predicciones de validación cruzada), **nunca** mirando el conjunto de prueba. Si lo eligiéramos con prueba, estaríamos haciendo trampa y la evaluación final sería optimista.

La siguiente celda *(ya resuelta)* obtiene, para cada paciente de entrenamiento, la probabilidad que le asigna un modelo que **no lo vio** durante su entrenamiento (`cross_val_predict`), y calcula la sensibilidad y especificidad para 91 umbrales entre 0.05 y 0.95.

In [ ]:
proba_cv = cross_val_predict(mejor_modelo, X_train, y_train, cv=validacion, method="predict_proba")[:, 1]

umbrales = np.round(np.arange(0.05, 0.951, 0.01), 2)
sens_cv  = np.array([recall_score(y_train, (proba_cv >= u).astype(int)) for u in umbrales])
espec_cv = np.array([recall_score(y_train, (proba_cv >= u).astype(int), pos_label=0) for u in umbrales])

SENSIBILIDAD_MINIMA = 0.90   # requisito del servicio de cardiología
graficar_umbral(umbrales, sens_cv, espec_cv, sens_minima=SENSIBILIDAD_MINIMA)

### ✏️ TODO 6 — Elige el umbral y aplícalo *(10 pts)*
El servicio de cardiología pide que el modelo detecte **al menos 90 de cada 100 enfermos** (`SENSIBILIDAD_MINIMA = 0.90`). Entre todos los umbrales que cumplen ese requisito, conviene el **más alto**, porque es el que conserva más especificidad.

1. `umbral_elegido`: el valor **más alto** de `umbrales` cuya `sens_cv` sea **≥ `SENSIBILIDAD_MINIMA`**.
2. `y_pred_umbral`: aplica ese umbral a las probabilidades de **prueba** del modelo optimizado (`y_proba_opt`) para obtener ceros y unos.
3. `metricas_umbral`: calcula sus métricas con tu función.

<details><summary>💡 Pista 1 — filtrar con NumPy</summary>

`umbrales[sens_cv >= SENSIBILIDAD_MINIMA]` te devuelve solo los umbrales que cumplen. ¿Cuál de ellos es el máximo?
</details>

<details><summary>💡 Pista 2 — aplicar el umbral</summary>

`(y_proba_opt >= umbral_elegido)` da `True/False`; conviértelo a enteros con `.astype(int)`.
</details>

In [ ]:
# ✏️ TODO 6 ─────────────────────────────────────────────────────────────
# ▼▼▼ TU CÓDIGO AQUÍ ▼▼▼
umbral_elegido  = None
y_pred_umbral   = None
metricas_umbral = None
# ▲▲▲ FIN DE TU CÓDIGO ▲▲▲

In [ ]:
# ✅ VERIFICACIÓN TODO 6
assert umbral_elegido is not None, "Define umbral_elegido."
_umbral_ok = umbrales[sens_cv >= SENSIBILIDAD_MINIMA].max()
assert np.isclose(umbral_elegido, _umbral_ok), \
    f"Tu umbral ({umbral_elegido}) no es el MÁS ALTO que cumple sensibilidad ≥ {SENSIBILIDAD_MINIMA} en validación."
assert y_pred_umbral is not None and set(np.unique(y_pred_umbral)) <= {0, 1}, "y_pred_umbral debe contener solo 0 y 1."
assert np.array_equal(y_pred_umbral, (y_proba_opt >= umbral_elegido).astype(int)), \
    "Aplica el umbral a y_proba_opt (probabilidades de PRUEBA del modelo optimizado)."
_i = int(np.argmin(np.abs(umbrales - umbral_elegido)))
print(f"✅ Umbral elegido: {umbral_elegido:.2f}   (en validación: sensibilidad {sens_cv[_i]:.3f}, especificidad {espec_cv[_i]:.3f})")
graficar_umbral(umbrales, sens_cv, espec_cv, umbral_elegido=umbral_elegido, sens_minima=SENSIBILIDAD_MINIMA)

In [ ]:
graficar_matrices_lado_a_lado([
    (y_test, y_pred_base,   "Base · umbral 0.50"),
    (y_test, y_pred_opt,    "Optimizado · umbral 0.50"),
    (y_test, y_pred_umbral, f"Optimizado · umbral {umbral_elegido:.2f}"),
])

In [ ]:
resultados_finales = {"Base": metricas_base, "Optimizado": metricas_opt,
                      f"Optimizado + umbral {umbral_elegido:.2f}": metricas_umbral}
display(tabla_comparativa(resultados_finales))
graficar_comparacion_metricas(resultados_finales, "Resumen final en el conjunto de prueba")

---
## 8 · Preguntas de análisis *(20 pts — 4 pts cada una)*
Responde en las celdas de texto usando **tus propios resultados** (cita números de tus tablas y matrices).

**P1.** Compara el modelo base contra el optimizado en prueba: ¿cuántos enfermos dejó de detectar cada uno (FN)? Usando la gráfica de la búsqueda y la *zona de empate*, ¿dirías que la mejora es confiable o podría deberse al azar? Explica.

**P2.** ¿Qué valores de `C` y `l1_ratio` eligió tu búsqueda? Explica con tus palabras qué significa ese nivel de regularización y qué variables clínicas eliminó (o no) el modelo según la gráfica de coeficientes.

**P3.** En el experimento de la trampa, ¿por qué optimizar solo la sensibilidad produjo un modelo inútil? ¿Qué dos métricas lo delatan de inmediato y por qué?

**P4.** Al bajar el umbral, ¿qué ganaste y qué perdiste (en número de pacientes)? ¿Recomendarías ese umbral para **triaje** (decidir quién pasa a estudios no invasivos)? ¿Y si el modelo decidiera directamente un **cateterismo cardiaco** (procedimiento invasivo con riesgos)?

**P5.** ¿Por qué elegimos el umbral con predicciones de validación cruzada sobre **entrenamiento** y no directamente con el conjunto de **prueba**?

**Respuesta P1:**

*Escribe aquí tu respuesta…*

**Respuesta P2:**

*Escribe aquí tu respuesta…*

**Respuesta P3:**

*Escribe aquí tu respuesta…*

**Respuesta P4:**

*Escribe aquí tu respuesta…*

**Respuesta P5:**

*Escribe aquí tu respuesta…*